In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from pathlib import Path
from PIL import Image
import numpy as np
import os
import json
from datetime import datetime

if torch.backends.mps.is_available():
    print("PyTorch sees the M4 Pro GPU (MPS).")
    device = torch.device("mps")
else:
    print("GPU not found. Double check your installation.")

Success! PyTorch sees the M4 Pro GPU (MPS).


In [ ]:
class KneeXrayDataset(Dataset):
    def __init__(self, image_dir, image_size=224):
        self.image_dir = Path(image_dir)
        extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
        self.image_paths = sorted([
            f for f in self.image_dir.rglob("*")
            if f.suffix.lower() in extensions
        ])
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),  # normalize to [-1, 1]
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")
        return self.transform(img)


class Generator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(noise_dim, 14 * 14 * 512),
            nn.BatchNorm1d(14 * 14 * 512),
            nn.LeakyReLU(0.2),
        )

        self.conv_blocks = nn.Sequential(
            # 14x14x512 → 28x28x256
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            # 28x28x256 → 56x56x128
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            # 56x56x128 → 112x112x64
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),

            # 112x112x64 → 224x224x32
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),

            # 224x224x32 → 224x224x1
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 512, 14, 14)
        return self.conv_blocks(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_blocks = nn.Sequential(
            # 224x224x1 → 112x112x32
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 112x112x32 → 56x56x64
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 56x56x64 → 28x28x128
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 28x28x128 → 14x14x256
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 1),
            nn.Sigmoid(),
        )

    def forward(self, img):
        features = self.conv_blocks(img)
        return self.classifier(features)


# ============================================================
# Training
# ============================================================
def get_device():
    if torch.backends.mps.is_available():
        print("Using: Apple MPS (Metal)")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("Using: CUDA GPU")
        return torch.device("cuda")
    else:
        print("Using: CPU (this will be slow)")
        return torch.device("cpu")


def train_dcgan(config):
    device = get_device()

    # Dataset
    dataset = KneeXrayDataset(config["input_dir"], config["image_size"])
    dataloader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=0,
        drop_last=True,
    )
    print(f"Dataset: {len(dataset)} images from {config['input_dir']}")

    # Models
    gen = Generator(config["noise_dim"]).to(device)
    disc = Discriminator().to(device)

    # Optimizers
    gen_opt = optim.Adam(gen.parameters(), lr=config["lr"], betas=(0.5, 0.999))
    disc_opt = optim.Adam(disc.parameters(), lr=config["lr"], betas=(0.5, 0.999))

    # Loss
    criterion = nn.BCELoss()

    # Output directory
    output_dir = Path(config["output_dir"])
    samples_dir = output_dir / "samples"
    checkpoints_dir = output_dir / "checkpoints"
    samples_dir.mkdir(parents=True, exist_ok=True)
    checkpoints_dir.mkdir(parents=True, exist_ok=True)

    # Fixed noise for tracking progress
    fixed_noise = torch.randn(16, config["noise_dim"], device=device)

    # Label smoothing for better training stability
    real_label_val = 0.9  # instead of 1.0
    fake_label_val = 0.0

    print(f"\nTraining for {config['epochs']} epochs...")
    print(f"Batch size: {config['batch_size']}")
    print(f"Learning rate: {config['lr']}\n")

    history = []

    for epoch in range(config["epochs"]):
        gen.train()
        disc.train()
        d_losses = []
        g_losses = []

        for batch_idx, real_images in enumerate(dataloader):
            batch_size = real_images.size(0)
            real_images = real_images.to(device)

            # Labels
            real_labels = torch.full((batch_size, 1), real_label_val, device=device)
            fake_labels = torch.full((batch_size, 1), fake_label_val, device=device)

            # ----- Train Discriminator -----
            disc_opt.zero_grad()

            # Real images
            real_output = disc(real_images)
            d_loss_real = criterion(real_output, real_labels)

            # Fake images
            noise = torch.randn(batch_size, config["noise_dim"], device=device)
            fake_images = gen(noise)
            fake_output = disc(fake_images.detach())
            d_loss_fake = criterion(fake_output, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            disc_opt.step()

            # ----- Train Generator -----
            gen_opt.zero_grad()

            fake_output = disc(fake_images)
            g_loss = criterion(fake_output, real_labels)  # generator wants disc to say "real"

            g_loss.backward()
            gen_opt.step()

            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())

        # Epoch stats
        avg_d = np.mean(d_losses)
        avg_g = np.mean(g_losses)
        history.append({"epoch": epoch + 1, "d_loss": avg_d, "g_loss": avg_g})

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch [{epoch+1}/{config['epochs']}] D_loss: {avg_d:.4f} | G_loss: {avg_g:.4f}")

        # Save sample images every 25 epochs
        if (epoch + 1) % 25 == 0 or epoch == 0:
            gen.eval()
            with torch.no_grad():
                samples = gen(fixed_noise).cpu()
                save_sample_grid(samples, samples_dir / f"epoch_{epoch+1:04d}.png")
            gen.train()

        # Save checkpoint every 50 epochs
        if (epoch + 1) % 50 == 0:
            torch.save({
                "epoch": epoch + 1,
                "generator": gen.state_dict(),
                "discriminator": disc.state_dict(),
                "gen_optimizer": gen_opt.state_dict(),
                "disc_optimizer": disc_opt.state_dict(),
            }, checkpoints_dir / f"checkpoint_epoch_{epoch+1:04d}.pt")

    # Save final models
    torch.save(gen.state_dict(), output_dir / "generator_final.pt")
    torch.save(disc.state_dict(), output_dir / "discriminator_final.pt")

    # Save training history
    with open(output_dir / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(f"\nTraining complete")
    print(f"  Final model: {output_dir / 'generator_final.pt'}")
    print(f"  Samples: {samples_dir}")

    return gen


def save_sample_grid(images, path, nrow=4):
    """Save a grid of generated images for visual inspection."""
    images = (images * 0.5 + 0.5).clamp(0, 1)  # denormalize from [-1,1] to [0,1]
    n = images.size(0)
    rows = (n + nrow - 1) // nrow

    grid_h = rows * 224
    grid_w = nrow * 224
    grid = np.ones((grid_h, grid_w), dtype=np.uint8) * 255

    for i in range(n):
        r, c = i // nrow, i % nrow
        img_np = (images[i, 0].numpy() * 255).astype(np.uint8)
        grid[r*224:(r+1)*224, c*224:(c+1)*224] = img_np

    from PIL import Image as PILImage
    PILImage.fromarray(grid).save(path)


# ============================================================
# Generation — use trained model to produce new images
# ============================================================
def generate_images(generator_path, output_dir, num_images, noise_dim=100, device=None):
    if device is None:
        device = get_device()

    gen = Generator(noise_dim).to(device)
    gen.load_state_dict(torch.load(generator_path, map_location=device))
    gen.eval()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Generating {num_images} images...")

    batch_size = 32
    count = 0

    with torch.no_grad():
        while count < num_images:
            current_batch = min(batch_size, num_images - count)
            noise = torch.randn(current_batch, noise_dim, device=device)
            fake_images = gen(noise)

            # Denormalize
            fake_images = (fake_images * 0.5 + 0.5).clamp(0, 1)

            for i in range(current_batch):
                img_np = (fake_images[i, 0].cpu().numpy() * 255).astype(np.uint8)
                img_pil = Image.fromarray(img_np)
                img_pil.save(output_dir / f"generated_{count:05d}.png")
                count += 1

            if count % 500 == 0:
                print(f"  [{count}/{num_images}] generated")

    print(f"Done! {count} images saved to {output_dir}")


# ============================================================
# Config — change these and run
# ============================================================
if __name__ == "__main__":

    # ---- STEP 1: TRAIN ----
    # Train one grade at a time. Start with Grade 4.
    CONFIG = {
        "input_dir": "dataset_augmented/4",    # ← grade folder with augmented images
        "output_dir": "dcgan_output/grade_4",  # ← where models and samples go
        "image_size": 224,
        "noise_dim": 100,
        "batch_size": 32,
        "lr": 0.0002,
        "epochs": 200,
    }

    trained_generator = train_dcgan(CONFIG)


Using: Apple MPS (Metal)
Dataset: 1864 images from dataset_augmented/4

Training for 200 epochs...
Batch size: 32
Learning rate: 0.0002

  Epoch [1/200] D_loss: 1.1593 | G_loss: 1.4391
  Epoch [10/200] D_loss: 1.2151 | G_loss: 1.0221
  Epoch [20/200] D_loss: 1.2816 | G_loss: 0.9442
  Epoch [30/200] D_loss: 1.2847 | G_loss: 0.9697
  Epoch [40/200] D_loss: 1.2498 | G_loss: 1.0265
  Epoch [50/200] D_loss: 1.2425 | G_loss: 1.0679
  Epoch [60/200] D_loss: 1.2121 | G_loss: 1.0686
  Epoch [70/200] D_loss: 1.2303 | G_loss: 1.0990
  Epoch [80/200] D_loss: 1.1988 | G_loss: 1.1197
  Epoch [90/200] D_loss: 1.2032 | G_loss: 1.1132
  Epoch [100/200] D_loss: 1.1848 | G_loss: 1.1266
  Epoch [110/200] D_loss: 1.1766 | G_loss: 1.1752
  Epoch [120/200] D_loss: 1.1589 | G_loss: 1.1855
  Epoch [130/200] D_loss: 1.1461 | G_loss: 1.1962
  Epoch [140/200] D_loss: 1.1523 | G_loss: 1.2098
  Epoch [150/200] D_loss: 1.0973 | G_loss: 1.4170
  Epoch [160/200] D_loss: 1.1024 | G_loss: 1.2471
  Epoch [170/200] D_loss

In [ ]:
generate_images("dcgan_output/grade_4/generator_final.pt", "./dcgan_generated/4", num_images=13136)

Using: Apple MPS (Metal)
Generating 13136 images...
  [4000/13136] generated
  [8000/13136] generated
  [12000/13136] generated
Done! 13136 images saved to dcgan_generated/4


In [ ]:


class KneeXrayDataset(Dataset):
    def __init__(self, image_dir, image_size=224):
        self.image_dir = Path(image_dir)
        extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
        self.image_paths = sorted([
            f for f in self.image_dir.rglob("*")
            if f.suffix.lower() in extensions
        ])
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),  # normalize to [-1, 1]
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")
        return self.transform(img)


class Generator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(noise_dim, 14 * 14 * 512),
            nn.BatchNorm1d(14 * 14 * 512),
            nn.LeakyReLU(0.2),
        )

        self.conv_blocks = nn.Sequential(
            # 14x14x512 → 28x28x256
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            # 28x28x256 → 56x56x128
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            # 56x56x128 → 112x112x64
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),

            # 112x112x64 → 224x224x32
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),

            # 224x224x32 → 224x224x1
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 512, 14, 14)
        return self.conv_blocks(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_blocks = nn.Sequential(
            # 224x224x1 → 112x112x32
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 112x112x32 → 56x56x64
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 56x56x64 → 28x28x128
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 28x28x128 → 14x14x256
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 1),
            nn.Sigmoid(),
        )

    def forward(self, img):
        features = self.conv_blocks(img)
        return self.classifier(features)


# ============================================================
# Training
# ============================================================
def get_device():
    if torch.backends.mps.is_available():
        print("Using: Apple MPS (Metal)")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("Using: CUDA GPU")
        return torch.device("cuda")
    else:
        print("Using: CPU (this will be slow)")
        return torch.device("cpu")


def train_dcgan(config):
    device = get_device()

    # Dataset
    dataset = KneeXrayDataset(config["input_dir"], config["image_size"])
    dataloader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=0,  # MPS works best with 0
        drop_last=True,
    )
    print(f"Dataset: {len(dataset)} images from {config['input_dir']}")

    # Models
    gen = Generator(config["noise_dim"]).to(device)
    disc = Discriminator().to(device)

    # Optimizers
    gen_opt = optim.Adam(gen.parameters(), lr=config["lr"], betas=(0.5, 0.999))
    disc_opt = optim.Adam(disc.parameters(), lr=config["lr"], betas=(0.5, 0.999))

    # Loss
    criterion = nn.BCELoss()

    # Output directory
    output_dir = Path(config["output_dir"])
    samples_dir = output_dir / "samples"
    checkpoints_dir = output_dir / "checkpoints"
    samples_dir.mkdir(parents=True, exist_ok=True)
    checkpoints_dir.mkdir(parents=True, exist_ok=True)

    # Fixed noise for tracking progress
    fixed_noise = torch.randn(16, config["noise_dim"], device=device)

    # Label smoothing for better training stability
    real_label_val = 0.9  # instead of 1.0
    fake_label_val = 0.0

    print(f"\nTraining for {config['epochs']} epochs...")
    print(f"Batch size: {config['batch_size']}")
    print(f"Learning rate: {config['lr']}\n")

    history = []

    for epoch in range(config["epochs"]):
        gen.train()
        disc.train()
        d_losses = []
        g_losses = []

        for batch_idx, real_images in enumerate(dataloader):
            batch_size = real_images.size(0)
            real_images = real_images.to(device)

            # Labels
            real_labels = torch.full((batch_size, 1), real_label_val, device=device)
            fake_labels = torch.full((batch_size, 1), fake_label_val, device=device)

            # ----- Train Discriminator -----
            disc_opt.zero_grad()

            # Real images
            real_output = disc(real_images)
            d_loss_real = criterion(real_output, real_labels)

            # Fake images
            noise = torch.randn(batch_size, config["noise_dim"], device=device)
            fake_images = gen(noise)
            fake_output = disc(fake_images.detach())
            d_loss_fake = criterion(fake_output, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            disc_opt.step()

            # ----- Train Generator -----
            gen_opt.zero_grad()

            fake_output = disc(fake_images)
            g_loss = criterion(fake_output, real_labels)  # generator wants disc to say "real"

            g_loss.backward()
            gen_opt.step()

            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())

        # Epoch stats
        avg_d = np.mean(d_losses)
        avg_g = np.mean(g_losses)
        history.append({"epoch": epoch + 1, "d_loss": avg_d, "g_loss": avg_g})

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch [{epoch+1}/{config['epochs']}] D_loss: {avg_d:.4f} | G_loss: {avg_g:.4f}")

        # Save sample images every 25 epochs
        if (epoch + 1) % 25 == 0 or epoch == 0:
            gen.eval()
            with torch.no_grad():
                samples = gen(fixed_noise).cpu()
                save_sample_grid(samples, samples_dir / f"epoch_{epoch+1:04d}.png")
            gen.train()

        # Save checkpoint every 50 epochs
        if (epoch + 1) % 50 == 0:
            torch.save({
                "epoch": epoch + 1,
                "generator": gen.state_dict(),
                "discriminator": disc.state_dict(),
                "gen_optimizer": gen_opt.state_dict(),
                "disc_optimizer": disc_opt.state_dict(),
            }, checkpoints_dir / f"checkpoint_epoch_{epoch+1:04d}.pt")

    # Save final models
    torch.save(gen.state_dict(), output_dir / "generator_final.pt")
    torch.save(disc.state_dict(), output_dir / "discriminator_final.pt")

    # Save training history
    with open(output_dir / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(f"\nTraining complete!")
    print(f"  Final model: {output_dir / 'generator_final.pt'}")
    print(f"  Samples: {samples_dir}")

    return gen


def save_sample_grid(images, path, nrow=4):
    images = (images * 0.5 + 0.5).clamp(0, 1)  # denormalize from [-1,1] to [0,1]
    n = images.size(0)
    rows = (n + nrow - 1) // nrow

    grid_h = rows * 224
    grid_w = nrow * 224
    grid = np.ones((grid_h, grid_w), dtype=np.uint8) * 255

    for i in range(n):
        r, c = i // nrow, i % nrow
        img_np = (images[i, 0].numpy() * 255).astype(np.uint8)
        grid[r*224:(r+1)*224, c*224:(c+1)*224] = img_np

    from PIL import Image as PILImage
    PILImage.fromarray(grid).save(path)


def generate_images(generator_path, output_dir, num_images, noise_dim=100, device=None):
    if device is None:
        device = get_device()

    gen = Generator(noise_dim).to(device)
    gen.load_state_dict(torch.load(generator_path, map_location=device))
    gen.eval()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Generating {num_images} images...")

    batch_size = 32
    count = 0

    with torch.no_grad():
        while count < num_images:
            current_batch = min(batch_size, num_images - count)
            noise = torch.randn(current_batch, noise_dim, device=device)
            fake_images = gen(noise)

            # Denormalize
            fake_images = (fake_images * 0.5 + 0.5).clamp(0, 1)

            for i in range(current_batch):
                img_np = (fake_images[i, 0].cpu().numpy() * 255).astype(np.uint8)
                img_pil = Image.fromarray(img_np)
                img_pil.save(output_dir / f"generated_{count:05d}.png")
                count += 1

            if count % 500 == 0:
                print(f"  [{count}/{num_images}] generated")

    print(f"Done! {count} images saved to {output_dir}")

if __name__ == "__main__":

    CONFIG = {
        "input_dir": "dataset_augmented/3",    
        "output_dir": "dcgan_output/grade_3",  
        "image_size": 224,
        "noise_dim": 100,
        "batch_size": 32,
        "lr": 0.0002,
        "epochs": 200,
    }

    trained_generator = train_dcgan(CONFIG)


Using: Apple MPS (Metal)
Dataset: 4548 images from dataset_augmented/3

Training for 200 epochs...
Batch size: 32
Learning rate: 0.0002

  Epoch [1/200] D_loss: 1.2955 | G_loss: 1.0328
  Epoch [10/200] D_loss: 1.2241 | G_loss: 1.0272
  Epoch [20/200] D_loss: 1.2772 | G_loss: 0.9645
  Epoch [30/200] D_loss: 1.2993 | G_loss: 0.9398
  Epoch [40/200] D_loss: 1.2226 | G_loss: 1.1505
  Epoch [50/200] D_loss: 1.2665 | G_loss: 1.0314
  Epoch [60/200] D_loss: 1.2861 | G_loss: 0.9815
  Epoch [70/200] D_loss: 1.2970 | G_loss: 0.9618
  Epoch [80/200] D_loss: 1.3032 | G_loss: 0.9337
  Epoch [90/200] D_loss: 1.3089 | G_loss: 0.9181
  Epoch [100/200] D_loss: 1.3096 | G_loss: 0.9271
  Epoch [110/200] D_loss: 1.3075 | G_loss: 0.9197
  Epoch [120/200] D_loss: 1.3098 | G_loss: 0.9116
  Epoch [130/200] D_loss: 1.3079 | G_loss: 0.9148
  Epoch [140/200] D_loss: 1.3021 | G_loss: 0.9147
  Epoch [150/200] D_loss: 1.3111 | G_loss: 0.9224
  Epoch [160/200] D_loss: 1.3041 | G_loss: 0.9208
  Epoch [170/200] D_loss

In [4]:


class KneeXrayDataset(Dataset):
    def __init__(self, image_dir, image_size=224):
        self.image_dir = Path(image_dir)
        extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
        self.image_paths = sorted([
            f for f in self.image_dir.rglob("*")
            if f.suffix.lower() in extensions
        ])
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),  # normalize to [-1, 1]
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")
        return self.transform(img)


class Generator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(noise_dim, 14 * 14 * 512),
            nn.BatchNorm1d(14 * 14 * 512),
            nn.LeakyReLU(0.2),
        )

        self.conv_blocks = nn.Sequential(
            # 14x14x512 → 28x28x256
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            # 28x28x256 → 56x56x128
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            # 56x56x128 → 112x112x64
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),

            # 112x112x64 → 224x224x32
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),

            # 224x224x32 → 224x224x1
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 512, 14, 14)
        return self.conv_blocks(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_blocks = nn.Sequential(
            # 224x224x1 → 112x112x32
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 112x112x32 → 56x56x64
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 56x56x64 → 28x28x128
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 28x28x128 → 14x14x256
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 1),
            nn.Sigmoid(),
        )

    def forward(self, img):
        features = self.conv_blocks(img)
        return self.classifier(features)


# ============================================================
# Training
# ============================================================
def get_device():
    if torch.backends.mps.is_available():
        print("Using: Apple MPS (Metal)")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("Using: CUDA GPU")
        return torch.device("cuda")
    else:
        print("Using: CPU (this will be slow)")
        return torch.device("cpu")


def train_dcgan(config):
    device = get_device()

    # Dataset
    dataset = KneeXrayDataset(config["input_dir"], config["image_size"])
    dataloader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=0,  # MPS works best with 0
        drop_last=True,
    )
    print(f"Dataset: {len(dataset)} images from {config['input_dir']}")

    # Models
    gen = Generator(config["noise_dim"]).to(device)
    disc = Discriminator().to(device)

    # Optimizers
    gen_opt = optim.Adam(gen.parameters(), lr=config["lr"], betas=(0.5, 0.999))
    disc_opt = optim.Adam(disc.parameters(), lr=config["lr"], betas=(0.5, 0.999))

    # Loss
    criterion = nn.BCELoss()

    # Output directory
    output_dir = Path(config["output_dir"])
    samples_dir = output_dir / "samples"
    checkpoints_dir = output_dir / "checkpoints"
    samples_dir.mkdir(parents=True, exist_ok=True)
    checkpoints_dir.mkdir(parents=True, exist_ok=True)

    # Fixed noise for tracking progress
    fixed_noise = torch.randn(16, config["noise_dim"], device=device)

    # Label smoothing for better training stability
    real_label_val = 0.9  # instead of 1.0
    fake_label_val = 0.0

    print(f"\nTraining for {config['epochs']} epochs...")
    print(f"Batch size: {config['batch_size']}")
    print(f"Learning rate: {config['lr']}\n")

    history = []

    for epoch in range(config["epochs"]):
        gen.train()
        disc.train()
        d_losses = []
        g_losses = []

        for batch_idx, real_images in enumerate(dataloader):
            batch_size = real_images.size(0)
            real_images = real_images.to(device)

            # Labels
            real_labels = torch.full((batch_size, 1), real_label_val, device=device)
            fake_labels = torch.full((batch_size, 1), fake_label_val, device=device)

            # ----- Train Discriminator -----
            disc_opt.zero_grad()

            # Real images
            real_output = disc(real_images)
            d_loss_real = criterion(real_output, real_labels)

            # Fake images
            noise = torch.randn(batch_size, config["noise_dim"], device=device)
            fake_images = gen(noise)
            fake_output = disc(fake_images.detach())
            d_loss_fake = criterion(fake_output, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            disc_opt.step()

            # ----- Train Generator -----
            gen_opt.zero_grad()

            fake_output = disc(fake_images)
            g_loss = criterion(fake_output, real_labels)  # generator wants disc to say "real"

            g_loss.backward()
            gen_opt.step()

            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())

        # Epoch stats
        avg_d = np.mean(d_losses)
        avg_g = np.mean(g_losses)
        history.append({"epoch": epoch + 1, "d_loss": avg_d, "g_loss": avg_g})

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch [{epoch+1}/{config['epochs']}] D_loss: {avg_d:.4f} | G_loss: {avg_g:.4f}")

        # Save sample images every 25 epochs
        if (epoch + 1) % 25 == 0 or epoch == 0:
            gen.eval()
            with torch.no_grad():
                samples = gen(fixed_noise).cpu()
                save_sample_grid(samples, samples_dir / f"epoch_{epoch+1:04d}.png")
            gen.train()

        # Save checkpoint every 50 epochs
        if (epoch + 1) % 50 == 0:
            torch.save({
                "epoch": epoch + 1,
                "generator": gen.state_dict(),
                "discriminator": disc.state_dict(),
                "gen_optimizer": gen_opt.state_dict(),
                "disc_optimizer": disc_opt.state_dict(),
            }, checkpoints_dir / f"checkpoint_epoch_{epoch+1:04d}.pt")

    # Save final models
    torch.save(gen.state_dict(), output_dir / "generator_final.pt")
    torch.save(disc.state_dict(), output_dir / "discriminator_final.pt")

    # Save training history
    with open(output_dir / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(f"\nTraining complete!")
    print(f"  Final model: {output_dir / 'generator_final.pt'}")
    print(f"  Samples: {samples_dir}")

    return gen


def save_sample_grid(images, path, nrow=4):
    """Save a grid of generated images for visual inspection."""
    images = (images * 0.5 + 0.5).clamp(0, 1)  # denormalize from [-1,1] to [0,1]
    n = images.size(0)
    rows = (n + nrow - 1) // nrow

    grid_h = rows * 224
    grid_w = nrow * 224
    grid = np.ones((grid_h, grid_w), dtype=np.uint8) * 255

    for i in range(n):
        r, c = i // nrow, i % nrow
        img_np = (images[i, 0].numpy() * 255).astype(np.uint8)
        grid[r*224:(r+1)*224, c*224:(c+1)*224] = img_np

    from PIL import Image as PILImage
    PILImage.fromarray(grid).save(path)


# ============================================================
# Generation — use trained model to produce new images
# ============================================================
def generate_images(generator_path, output_dir, num_images, noise_dim=100, device=None):
    if device is None:
        device = get_device()

    gen = Generator(noise_dim).to(device)
    gen.load_state_dict(torch.load(generator_path, map_location=device))
    gen.eval()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Generating {num_images} images...")

    batch_size = 32
    count = 0

    with torch.no_grad():
        while count < num_images:
            current_batch = min(batch_size, num_images - count)
            noise = torch.randn(current_batch, noise_dim, device=device)
            fake_images = gen(noise)

            # Denormalize
            fake_images = (fake_images * 0.5 + 0.5).clamp(0, 1)

            for i in range(current_batch):
                img_np = (fake_images[i, 0].cpu().numpy() * 255).astype(np.uint8)
                img_pil = Image.fromarray(img_np)
                img_pil.save(output_dir / f"generated_{count:05d}.png")
                count += 1

            if count % 500 == 0:
                print(f"  [{count}/{num_images}] generated")

    print(f"Done! {count} images saved to {output_dir}")


# ============================================================
# Config — change these and run
# ============================================================
if __name__ == "__main__":

    # ---- STEP 1: TRAIN ----
    # Train one grade at a time. Start with Grade 4.
    CONFIG = {
        "input_dir": "dataset_augmented/2",    # ← grade folder with augmented images
        "output_dir": "dcgan_output/grade_2",  # ← where models and samples go
        "image_size": 224,
        "noise_dim": 100,
        "batch_size": 32,
        "lr": 0.0002,
        "epochs": 200,
    }

    trained_generator = train_dcgan(CONFIG)

    # ---- STEP 2: GENERATE ----
    # After training looks good (check samples folder), generate images.
    # Uncomment below when ready:

    # generate_images(
    #     generator_path="./dcgan_output/grade_4/generator_final.pt",
    #     output_dir="./dcgan_generated/4",
    #     num_images=5000,  # ← how many synthetic images you want
    # )

Using: Apple MPS (Metal)
Dataset: 8872 images from dataset_augmented/2

Training for 200 epochs...
Batch size: 32
Learning rate: 0.0002

  Epoch [1/200] D_loss: 1.3037 | G_loss: 0.9791
  Epoch [10/200] D_loss: 1.3122 | G_loss: 0.8992
  Epoch [20/200] D_loss: 1.3231 | G_loss: 0.8858
  Epoch [30/200] D_loss: 1.3280 | G_loss: 0.8840
  Epoch [40/200] D_loss: 1.3326 | G_loss: 0.8715
  Epoch [50/200] D_loss: 1.3401 | G_loss: 0.8614
  Epoch [60/200] D_loss: 1.3398 | G_loss: 0.8614
  Epoch [70/200] D_loss: 1.3447 | G_loss: 0.8557
  Epoch [80/200] D_loss: 1.3402 | G_loss: 0.8535
  Epoch [90/200] D_loss: 1.3418 | G_loss: 0.8541
  Epoch [100/200] D_loss: 1.3439 | G_loss: 0.8484
  Epoch [110/200] D_loss: 1.3433 | G_loss: 0.8521
  Epoch [120/200] D_loss: 1.3431 | G_loss: 0.8532
  Epoch [130/200] D_loss: 1.3439 | G_loss: 0.8515
  Epoch [140/200] D_loss: 1.3404 | G_loss: 0.8542
  Epoch [150/200] D_loss: 1.3400 | G_loss: 0.8525
  Epoch [160/200] D_loss: 1.3403 | G_loss: 0.8553
  Epoch [170/200] D_loss

In [5]:


class KneeXrayDataset(Dataset):
    def __init__(self, image_dir, image_size=224):
        self.image_dir = Path(image_dir)
        extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
        self.image_paths = sorted([
            f for f in self.image_dir.rglob("*")
            if f.suffix.lower() in extensions
        ])
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),  # normalize to [-1, 1]
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")
        return self.transform(img)


class Generator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(noise_dim, 14 * 14 * 512),
            nn.BatchNorm1d(14 * 14 * 512),
            nn.LeakyReLU(0.2),
        )

        self.conv_blocks = nn.Sequential(
            # 14x14x512 → 28x28x256
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            # 28x28x256 → 56x56x128
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            # 56x56x128 → 112x112x64
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),

            # 112x112x64 → 224x224x32
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),

            # 224x224x32 → 224x224x1
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 512, 14, 14)
        return self.conv_blocks(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_blocks = nn.Sequential(
            # 224x224x1 → 112x112x32
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 112x112x32 → 56x56x64
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 56x56x64 → 28x28x128
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 28x28x128 → 14x14x256
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 1),
            nn.Sigmoid(),
        )

    def forward(self, img):
        features = self.conv_blocks(img)
        return self.classifier(features)


# ============================================================
# Training
# ============================================================
def get_device():
    if torch.backends.mps.is_available():
        print("Using: Apple MPS (Metal)")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("Using: CUDA GPU")
        return torch.device("cuda")
    else:
        print("Using: CPU (this will be slow)")
        return torch.device("cpu")


def train_dcgan(config):
    device = get_device()

    # Dataset
    dataset = KneeXrayDataset(config["input_dir"], config["image_size"])
    dataloader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=0,  # MPS works best with 0
        drop_last=True,
    )
    print(f"Dataset: {len(dataset)} images from {config['input_dir']}")

    # Models
    gen = Generator(config["noise_dim"]).to(device)
    disc = Discriminator().to(device)

    # Optimizers
    gen_opt = optim.Adam(gen.parameters(), lr=config["lr"], betas=(0.5, 0.999))
    disc_opt = optim.Adam(disc.parameters(), lr=config["lr"], betas=(0.5, 0.999))

    # Loss
    criterion = nn.BCELoss()

    # Output directory
    output_dir = Path(config["output_dir"])
    samples_dir = output_dir / "samples"
    checkpoints_dir = output_dir / "checkpoints"
    samples_dir.mkdir(parents=True, exist_ok=True)
    checkpoints_dir.mkdir(parents=True, exist_ok=True)

    # Fixed noise for tracking progress
    fixed_noise = torch.randn(16, config["noise_dim"], device=device)

    # Label smoothing for better training stability
    real_label_val = 0.9  # instead of 1.0
    fake_label_val = 0.0

    print(f"\nTraining for {config['epochs']} epochs...")
    print(f"Batch size: {config['batch_size']}")
    print(f"Learning rate: {config['lr']}\n")

    history = []

    for epoch in range(config["epochs"]):
        gen.train()
        disc.train()
        d_losses = []
        g_losses = []

        for batch_idx, real_images in enumerate(dataloader):
            batch_size = real_images.size(0)
            real_images = real_images.to(device)

            # Labels
            real_labels = torch.full((batch_size, 1), real_label_val, device=device)
            fake_labels = torch.full((batch_size, 1), fake_label_val, device=device)

            # ----- Train Discriminator -----
            disc_opt.zero_grad()

            # Real images
            real_output = disc(real_images)
            d_loss_real = criterion(real_output, real_labels)

            # Fake images
            noise = torch.randn(batch_size, config["noise_dim"], device=device)
            fake_images = gen(noise)
            fake_output = disc(fake_images.detach())
            d_loss_fake = criterion(fake_output, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            disc_opt.step()

            # ----- Train Generator -----
            gen_opt.zero_grad()

            fake_output = disc(fake_images)
            g_loss = criterion(fake_output, real_labels)  # generator wants disc to say "real"

            g_loss.backward()
            gen_opt.step()

            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())

        # Epoch stats
        avg_d = np.mean(d_losses)
        avg_g = np.mean(g_losses)
        history.append({"epoch": epoch + 1, "d_loss": avg_d, "g_loss": avg_g})

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch [{epoch+1}/{config['epochs']}] D_loss: {avg_d:.4f} | G_loss: {avg_g:.4f}")

        # Save sample images every 25 epochs
        if (epoch + 1) % 25 == 0 or epoch == 0:
            gen.eval()
            with torch.no_grad():
                samples = gen(fixed_noise).cpu()
                save_sample_grid(samples, samples_dir / f"epoch_{epoch+1:04d}.png")
            gen.train()

        # Save checkpoint every 50 epochs
        if (epoch + 1) % 50 == 0:
            torch.save({
                "epoch": epoch + 1,
                "generator": gen.state_dict(),
                "discriminator": disc.state_dict(),
                "gen_optimizer": gen_opt.state_dict(),
                "disc_optimizer": disc_opt.state_dict(),
            }, checkpoints_dir / f"checkpoint_epoch_{epoch+1:04d}.pt")

    # Save final models
    torch.save(gen.state_dict(), output_dir / "generator_final.pt")
    torch.save(disc.state_dict(), output_dir / "discriminator_final.pt")

    # Save training history
    with open(output_dir / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(f"\nTraining complete!")
    print(f"  Final model: {output_dir / 'generator_final.pt'}")
    print(f"  Samples: {samples_dir}")

    return gen


def save_sample_grid(images, path, nrow=4):
    """Save a grid of generated images for visual inspection."""
    images = (images * 0.5 + 0.5).clamp(0, 1)  # denormalize from [-1,1] to [0,1]
    n = images.size(0)
    rows = (n + nrow - 1) // nrow

    grid_h = rows * 224
    grid_w = nrow * 224
    grid = np.ones((grid_h, grid_w), dtype=np.uint8) * 255

    for i in range(n):
        r, c = i // nrow, i % nrow
        img_np = (images[i, 0].numpy() * 255).astype(np.uint8)
        grid[r*224:(r+1)*224, c*224:(c+1)*224] = img_np

    from PIL import Image as PILImage
    PILImage.fromarray(grid).save(path)


# ============================================================
# Generation — use trained model to produce new images
# ============================================================
def generate_images(generator_path, output_dir, num_images, noise_dim=100, device=None):
    if device is None:
        device = get_device()

    gen = Generator(noise_dim).to(device)
    gen.load_state_dict(torch.load(generator_path, map_location=device))
    gen.eval()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Generating {num_images} images...")

    batch_size = 32
    count = 0

    with torch.no_grad():
        while count < num_images:
            current_batch = min(batch_size, num_images - count)
            noise = torch.randn(current_batch, noise_dim, device=device)
            fake_images = gen(noise)

            # Denormalize
            fake_images = (fake_images * 0.5 + 0.5).clamp(0, 1)

            for i in range(current_batch):
                img_np = (fake_images[i, 0].cpu().numpy() * 255).astype(np.uint8)
                img_pil = Image.fromarray(img_np)
                img_pil.save(output_dir / f"generated_{count:05d}.png")
                count += 1

            if count % 500 == 0:
                print(f"  [{count}/{num_images}] generated")

    print(f"Done! {count} images saved to {output_dir}")


# ============================================================
# Config — change these and run
# ============================================================
if __name__ == "__main__":

    # ---- STEP 1: TRAIN ----
    # Train one grade at a time. Start with Grade 4.
    CONFIG = {
        "input_dir": "dataset_augmented/1",    # ← grade folder with augmented images
        "output_dir": "dcgan_output/grade_1",  # ← where models and samples go
        "image_size": 224,
        "noise_dim": 100,
        "batch_size": 32,
        "lr": 0.0002,
        "epochs": 200,
    }

    trained_generator = train_dcgan(CONFIG)

    # ---- STEP 2: GENERATE ----
    # After training looks good (check samples folder), generate images.
    # Uncomment below when ready:

    # generate_images(
    #     generator_path="./dcgan_output/grade_4/generator_final.pt",
    #     output_dir="./dcgan_generated/4",
    #     num_images=5000,  # ← how many synthetic images you want
    # )

Using: Apple MPS (Metal)
Dataset: 6260 images from dataset_augmented/1

Training for 200 epochs...
Batch size: 32
Learning rate: 0.0002

  Epoch [1/200] D_loss: 1.2954 | G_loss: 0.9899
  Epoch [10/200] D_loss: 1.2884 | G_loss: 0.9326
  Epoch [20/200] D_loss: 1.3082 | G_loss: 0.9129
  Epoch [30/200] D_loss: 1.3154 | G_loss: 0.8915
  Epoch [40/200] D_loss: 1.2488 | G_loss: 1.1138
  Epoch [50/200] D_loss: 1.2743 | G_loss: 1.0056
  Epoch [60/200] D_loss: 1.3037 | G_loss: 0.9435
  Epoch [70/200] D_loss: 1.3180 | G_loss: 0.8944
  Epoch [80/200] D_loss: 1.3236 | G_loss: 0.9082
  Epoch [90/200] D_loss: 1.3279 | G_loss: 0.8915
  Epoch [100/200] D_loss: 1.3300 | G_loss: 0.8742
  Epoch [110/200] D_loss: 1.3334 | G_loss: 0.8736
  Epoch [120/200] D_loss: 1.3320 | G_loss: 0.8713
  Epoch [130/200] D_loss: 1.3298 | G_loss: 0.8707
  Epoch [140/200] D_loss: 1.3288 | G_loss: 0.8725
  Epoch [150/200] D_loss: 1.3296 | G_loss: 0.8723
  Epoch [160/200] D_loss: 1.3263 | G_loss: 0.8719
  Epoch [170/200] D_loss

In [6]:


class KneeXrayDataset(Dataset):
    def __init__(self, image_dir, image_size=224):
        self.image_dir = Path(image_dir)
        extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}
        self.image_paths = sorted([
            f for f in self.image_dir.rglob("*")
            if f.suffix.lower() in extensions
        ])
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),  # normalize to [-1, 1]
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")
        return self.transform(img)


class Generator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Linear(noise_dim, 14 * 14 * 512),
            nn.BatchNorm1d(14 * 14 * 512),
            nn.LeakyReLU(0.2),
        )

        self.conv_blocks = nn.Sequential(
            # 14x14x512 → 28x28x256
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),

            # 28x28x256 → 56x56x128
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            # 56x56x128 → 112x112x64
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),

            # 112x112x64 → 224x224x32
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),

            # 224x224x32 → 224x224x1
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 512, 14, 14)
        return self.conv_blocks(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_blocks = nn.Sequential(
            # 224x224x1 → 112x112x32
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 112x112x32 → 56x56x64
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 56x56x64 → 28x28x128
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),

            # 28x28x128 → 14x14x256
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 1),
            nn.Sigmoid(),
        )

    def forward(self, img):
        features = self.conv_blocks(img)
        return self.classifier(features)


# ============================================================
# Training
# ============================================================
def get_device():
    if torch.backends.mps.is_available():
        print("Using: Apple MPS (Metal)")
        return torch.device("mps")
    elif torch.cuda.is_available():
        print("Using: CUDA GPU")
        return torch.device("cuda")
    else:
        print("Using: CPU (this will be slow)")
        return torch.device("cpu")


def train_dcgan(config):
    device = get_device()

    # Dataset
    dataset = KneeXrayDataset(config["input_dir"], config["image_size"])
    dataloader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=0,  # MPS works best with 0
        drop_last=True,
    )
    print(f"Dataset: {len(dataset)} images from {config['input_dir']}")

    # Models
    gen = Generator(config["noise_dim"]).to(device)
    disc = Discriminator().to(device)

    # Optimizers
    gen_opt = optim.Adam(gen.parameters(), lr=config["lr"], betas=(0.5, 0.999))
    disc_opt = optim.Adam(disc.parameters(), lr=config["lr"], betas=(0.5, 0.999))

    # Loss
    criterion = nn.BCELoss()

    # Output directory
    output_dir = Path(config["output_dir"])
    samples_dir = output_dir / "samples"
    checkpoints_dir = output_dir / "checkpoints"
    samples_dir.mkdir(parents=True, exist_ok=True)
    checkpoints_dir.mkdir(parents=True, exist_ok=True)

    # Fixed noise for tracking progress
    fixed_noise = torch.randn(16, config["noise_dim"], device=device)

    # Label smoothing for better training stability
    real_label_val = 0.9  # instead of 1.0
    fake_label_val = 0.0

    print(f"\nTraining for {config['epochs']} epochs...")
    print(f"Batch size: {config['batch_size']}")
    print(f"Learning rate: {config['lr']}\n")

    history = []

    for epoch in range(config["epochs"]):
        gen.train()
        disc.train()
        d_losses = []
        g_losses = []

        for batch_idx, real_images in enumerate(dataloader):
            batch_size = real_images.size(0)
            real_images = real_images.to(device)

            # Labels
            real_labels = torch.full((batch_size, 1), real_label_val, device=device)
            fake_labels = torch.full((batch_size, 1), fake_label_val, device=device)

            # ----- Train Discriminator -----
            disc_opt.zero_grad()

            # Real images
            real_output = disc(real_images)
            d_loss_real = criterion(real_output, real_labels)

            # Fake images
            noise = torch.randn(batch_size, config["noise_dim"], device=device)
            fake_images = gen(noise)
            fake_output = disc(fake_images.detach())
            d_loss_fake = criterion(fake_output, fake_labels)

            d_loss = d_loss_real + d_loss_fake
            d_loss.backward()
            disc_opt.step()

            # ----- Train Generator -----
            gen_opt.zero_grad()

            fake_output = disc(fake_images)
            g_loss = criterion(fake_output, real_labels)  # generator wants disc to say "real"

            g_loss.backward()
            gen_opt.step()

            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())

        # Epoch stats
        avg_d = np.mean(d_losses)
        avg_g = np.mean(g_losses)
        history.append({"epoch": epoch + 1, "d_loss": avg_d, "g_loss": avg_g})

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch [{epoch+1}/{config['epochs']}] D_loss: {avg_d:.4f} | G_loss: {avg_g:.4f}")

        # Save sample images every 25 epochs
        if (epoch + 1) % 25 == 0 or epoch == 0:
            gen.eval()
            with torch.no_grad():
                samples = gen(fixed_noise).cpu()
                save_sample_grid(samples, samples_dir / f"epoch_{epoch+1:04d}.png")
            gen.train()

        # Save checkpoint every 50 epochs
        if (epoch + 1) % 50 == 0:
            torch.save({
                "epoch": epoch + 1,
                "generator": gen.state_dict(),
                "discriminator": disc.state_dict(),
                "gen_optimizer": gen_opt.state_dict(),
                "disc_optimizer": disc_opt.state_dict(),
            }, checkpoints_dir / f"checkpoint_epoch_{epoch+1:04d}.pt")

    # Save final models
    torch.save(gen.state_dict(), output_dir / "generator_final.pt")
    torch.save(disc.state_dict(), output_dir / "discriminator_final.pt")

    # Save training history
    with open(output_dir / "training_history.json", "w") as f:
        json.dump(history, f, indent=2)

    print(f"\nTraining complete!")
    print(f"  Final model: {output_dir / 'generator_final.pt'}")
    print(f"  Samples: {samples_dir}")

    return gen


def save_sample_grid(images, path, nrow=4):
    """Save a grid of generated images for visual inspection."""
    images = (images * 0.5 + 0.5).clamp(0, 1)  # denormalize from [-1,1] to [0,1]
    n = images.size(0)
    rows = (n + nrow - 1) // nrow

    grid_h = rows * 224
    grid_w = nrow * 224
    grid = np.ones((grid_h, grid_w), dtype=np.uint8) * 255

    for i in range(n):
        r, c = i // nrow, i % nrow
        img_np = (images[i, 0].numpy() * 255).astype(np.uint8)
        grid[r*224:(r+1)*224, c*224:(c+1)*224] = img_np

    from PIL import Image as PILImage
    PILImage.fromarray(grid).save(path)


# ============================================================
# Generation — use trained model to produce new images
# ============================================================
def generate_images(generator_path, output_dir, num_images, noise_dim=100, device=None):
    if device is None:
        device = get_device()

    gen = Generator(noise_dim).to(device)
    gen.load_state_dict(torch.load(generator_path, map_location=device))
    gen.eval()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Generating {num_images} images...")

    batch_size = 32
    count = 0

    with torch.no_grad():
        while count < num_images:
            current_batch = min(batch_size, num_images - count)
            noise = torch.randn(current_batch, noise_dim, device=device)
            fake_images = gen(noise)

            # Denormalize
            fake_images = (fake_images * 0.5 + 0.5).clamp(0, 1)

            for i in range(current_batch):
                img_np = (fake_images[i, 0].cpu().numpy() * 255).astype(np.uint8)
                img_pil = Image.fromarray(img_np)
                img_pil.save(output_dir / f"generated_{count:05d}.png")
                count += 1

            if count % 500 == 0:
                print(f"  [{count}/{num_images}] generated")

    print(f"Done! {count} images saved to {output_dir}")


# ============================================================
# Config — change these and run
# ============================================================
if __name__ == "__main__":

    # ---- STEP 1: TRAIN ----
    # Train one grade at a time. Start with Grade 4.
    CONFIG = {
        "input_dir": "dataset_augmented/0",    # ← grade folder with augmented images
        "output_dir": "dcgan_output/grade_0",  # ← where models and samples go
        "image_size": 224,
        "noise_dim": 100,
        "batch_size": 32,
        "lr": 0.0002,
        "epochs": 200,
    }

    trained_generator = train_dcgan(CONFIG)

    # ---- STEP 2: GENERATE ----
    # After training looks good (check samples folder), generate images.
    # Uncomment below when ready:

    # generate_images(
    #     generator_path="./dcgan_output/grade_4/generator_final.pt",
    #     output_dir="./dcgan_generated/4",
    #     num_images=5000,  # ← how many synthetic images you want
    # )

Using: Apple MPS (Metal)
Dataset: 13192 images from dataset_augmented/0

Training for 200 epochs...
Batch size: 32
Learning rate: 0.0002

  Epoch [1/200] D_loss: 1.2613 | G_loss: 1.0399
  Epoch [10/200] D_loss: 1.3147 | G_loss: 0.9180
  Epoch [20/200] D_loss: 1.3298 | G_loss: 0.8988
  Epoch [30/200] D_loss: 1.3467 | G_loss: 0.8585
  Epoch [40/200] D_loss: 1.3497 | G_loss: 0.8481
  Epoch [50/200] D_loss: 1.3516 | G_loss: 0.8453
  Epoch [60/200] D_loss: 1.3531 | G_loss: 0.8432
  Epoch [70/200] D_loss: 1.3545 | G_loss: 0.8395
  Epoch [80/200] D_loss: 1.3547 | G_loss: 0.8413
  Epoch [90/200] D_loss: 1.3542 | G_loss: 0.8384
  Epoch [100/200] D_loss: 1.3534 | G_loss: 0.8390
  Epoch [110/200] D_loss: 1.3540 | G_loss: 0.8376
  Epoch [120/200] D_loss: 1.3496 | G_loss: 0.8400
  Epoch [130/200] D_loss: 1.3520 | G_loss: 0.8397
  Epoch [140/200] D_loss: 1.3521 | G_loss: 0.8406
  Epoch [150/200] D_loss: 1.3522 | G_loss: 0.8371
  Epoch [160/200] D_loss: 1.3543 | G_loss: 0.8394
  Epoch [170/200] D_los

In [ ]:
generate_images("dcgan_output/grade_0/generator_final.pt", "./dcgan_generated/0", num_images=1808)
generate_images("dcgan_output/grade_1/generator_final.pt", "./dcgan_generated/1", num_images=8740)
generate_images("dcgan_output/grade_2/generator_final.pt", "./dcgan_generated/2", num_images=6128)
generate_images("dcgan_output/grade_3/generator_final.pt", "./dcgan_generated/3", num_images=10452)

Using: Apple MPS (Metal)
Generating 1808 images...
Done! 1808 images saved to dcgan_generated/0
Using: Apple MPS (Metal)
Generating 8740 images...
  [4000/8740] generated
  [8000/8740] generated
Done! 8740 images saved to dcgan_generated/1
Using: Apple MPS (Metal)
Generating 6128 images...
  [4000/6128] generated
Done! 6128 images saved to dcgan_generated/2
Using: Apple MPS (Metal)
Generating 10452 images...
  [4000/10452] generated
  [8000/10452] generated
Done! 10452 images saved to dcgan_generated/3


In [5]:
import torch
import numpy as np
from pathlib import Path
from PIL import Image
import torch.nn as nn

# ── Copy the Generator class here ──────────────────────────────
class Generator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(noise_dim, 14 * 14 * 512),
            nn.BatchNorm1d(14 * 14 * 512),
            nn.LeakyReLU(0.2),
        )
        self.conv_blocks = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 512, 14, 14)
        return self.conv_blocks(x)


def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def generate_images(generator_path, output_dir, num_images, noise_dim=100):
    device = get_device()
    gen = Generator(noise_dim).to(device)
    gen.load_state_dict(torch.load(generator_path, map_location=device))
    gen.eval()

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    with torch.no_grad():
        for i in range(num_images):
            noise = torch.randn(1, noise_dim, device=device)
            img = gen(noise)
            img = (img * 0.5 + 0.5).clamp(0, 1)
            img_np = (img[0, 0].cpu().numpy() * 255).astype(np.uint8)
            Image.fromarray(img_np).save(output_dir / f"generated_{i:05d}.png")
            print(f"Saved {i+1}/{num_images}")


# ── Run ────────────────────────────────────────────────────────
generate_images("dcgan_output/grade_0/generator_final.pt", "./dcgan_generatedd/0", num_images=2)
generate_images("dcgan_output/grade_1/generator_final.pt", "./dcgan_generatedd/1", num_images=2)
generate_images("dcgan_output/grade_2/generator_final.pt", "./dcgan_generatedd/2", num_images=3)
generate_images("dcgan_output/grade_3/generator_final.pt", "./dcgan_generatedd/3", num_images=2)
generate_images("dcgan_output/grade_4/generator_final.pt", "./dcgan_generatedd/4", num_images=2)

Saved 1/2
Saved 2/2
Saved 1/2
Saved 2/2
Saved 1/3
Saved 2/3
Saved 3/3
Saved 1/2
Saved 2/2
Saved 1/2
Saved 2/2
